# Eksperimen #4 — Tweak murah pada meanstd D2

Sebelum menimbang fine-tune YAMNet (butuh Kaggle GPU), habiskan dulu opsi murah di CPU.
Semua di atas fitur **meanstd (2048-d)** terbaik, recipe esf1. Ekspektasi rendah (pola
menunjukkan masalah representasi), tapi murah.

Tiga varian diuji vs baseline meanstd (**0.7983**):
- **cw** — `class_weight` (bobot lebih ke kelas lemah spt ambulance)
- **ens** — ensemble 5 head beda seed, rata-ratakan probabilitas
- **cw+ens** — gabungan

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ML = ROOT / "ml"

data = np.load(ML / "cache" / "yamnet_stats_d2.npz", allow_pickle=True)
emb_by_file = {fn: e for fn, e in zip(data["filename"], data["emb"])}
CLASSES = sorted(set(data["label"])); idx = {c: i for i, c in enumerate(CLASSES)}


def load(name):
    df = pd.read_csv(ML / f"split_d2_{name}.csv")
    X = np.stack([emb_by_file[fn] for fn in df.filename]).astype(np.float32)
    return X, df.label.map(idx).to_numpy()


Xtr, ytr = load("train"); Xva, yva = load("val"); Xte, yte = load("test")
mean = Xtr.mean(0, keepdims=True); std = Xtr.std(0, keepdims=True)
std = np.where(std < 1e-6, 1.0, std)
Xtr = (Xtr - mean) / std; Xva = (Xva - mean) / std; Xte = (Xte - mean) / std
print(f"train {Xtr.shape} · val {Xva.shape[0]} · test {Xte.shape[0]} · kelas {CLASSES}")

train (1195, 2048) · val 240 · test 240 · kelas ['ambulance', 'firetruck', 'police', 'traffic']


In [2]:
class ValMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, Xv, yv):
        super().__init__(); self.Xv, self.yv = Xv, yv
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = self.model.predict(self.Xv, verbose=0).argmax(1)
        logs["val_macro_f1"] = f1_score(self.yv, p, average="macro")


def build():
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(Xtr.shape[1],)),
        tf.keras.layers.Dense(256, activation="relu"), tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(len(CLASSES), activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def train_one(seed, class_weight=None):
    """Latih satu head, return probabilitas test (softmax)."""
    tf.keras.utils.set_random_seed(seed)
    m = build()
    cbs = [ValMacroF1(Xva, yva),
           tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                            patience=20, restore_best_weights=True),
           tf.keras.callbacks.ReduceLROnPlateau(monitor="val_macro_f1", mode="max",
                                                factor=0.5, patience=8, min_lr=1e-5)]
    m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=200, batch_size=32,
          callbacks=cbs, verbose=0, class_weight=class_weight)
    return m.predict(Xte, verbose=0)


cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
print("class_weight:", {CLASSES[k]: round(v, 3) for k, v in cw.items()})

class_weight: {'ambulance': 1.045, 'firetruck': 1.052, 'police': 0.922, 'traffic': 0.993}


In [3]:
def evaluate(name, proba):
    yp = proba.argmax(1)
    f1 = f1_score(yte, yp, average="macro"); acc = accuracy_score(yte, yp)
    per = dict(zip(CLASSES, f1_score(yte, yp, average=None)))
    print(f"[{name}] macro-F1={f1:.4f} acc={acc:.4f} · " +
          " ".join(f"{c}={per[c]:.3f}" for c in CLASSES))
    return f1


SEEDS = [42, 1, 2, 3, 4]
print("melatih varian ...\n")

# cw: satu head dgn class_weight
p_cw = train_one(42, class_weight=cw)
f_cw = evaluate("cw       ", p_cw)

# ens: rata-rata 5 head tanpa class_weight
p_ens = np.mean([train_one(s) for s in SEEDS], axis=0)
f_ens = evaluate("ens      ", p_ens)

# cw+ens: rata-rata 5 head dgn class_weight
p_cwe = np.mean([train_one(s, class_weight=cw) for s in SEEDS], axis=0)
f_cwe = evaluate("cw+ens   ", p_cwe)

melatih varian ...



[cw       ] macro-F1=0.7834 acc=0.7833 · ambulance=0.707 firetruck=0.729 police=0.748 traffic=0.950


[ens      ] macro-F1=0.7651 acc=0.7667 · ambulance=0.661 firetruck=0.700 police=0.724 traffic=0.975


[cw+ens   ] macro-F1=0.7868 acc=0.7875 · ambulance=0.719 firetruck=0.711 police=0.742 traffic=0.975


## Perbandingan

In [4]:
print(f"{'varian':12s} {'macro-F1':>9s} {'Δ vs meanstd':>13s}")
base = 0.7983
for name, f in [("meanstd", base), ("cw", f_cw), ("ens", f_ens), ("cw+ens", f_cwe)]:
    d = "" if name == "meanstd" else f"{f - base:+.4f}"
    print(f"{name:12s} {f:>9.4f} {d:>13s}")

best = max([("cw", f_cw), ("ens", f_ens), ("cw+ens", f_cwe)], key=lambda x: x[1])
print(f"\nvarian termurah terbaik: {best[0]} = {best[1]:.4f}")
print("catatan: tidak dicatat ke experiments.csv kecuali menang jelas — ini uji cepat.")

varian        macro-F1  Δ vs meanstd
meanstd         0.7983              
cw              0.7834       -0.0149
ens             0.7651       -0.0332
cw+ens          0.7868       -0.0115

varian termurah terbaik: cw+ens = 0.7868
catatan: tidak dicatat ke experiments.csv kecuali menang jelas — ini uji cepat.


---

**Interpretasi:** kalau salah satu varian menembus/mendekati 0.85, adopsi. Kalau semua
flat (±0.01 dari 0.7983), ini menegaskan plateau fitur beku → fine-tune YAMNet (Kaggle)
adalah satu-satunya lever tersisa untuk mengejar target.